# M1: Google's Gen AI Models — Google Gen AI SDK ハンズオンデモ

本ノートブックは、Gemini Enterprise Agent Platform (旧 Vertex AI) 上で Google Gen AI SDK (`google-genai`) を使い、Gemini モデルをひととおり動かしてみるハンズオン教材です。基本のテキスト生成から、マルチモーダル入力、マルチターン会話、ストリーミング、構造化出力、Function Calling、Grounding、画像生成、動画生成まで、実務でよく使う API 呼び出しパターンを順番に体験します。

すべてのコードは `from google import genai` / `from google.genai import types` という Google Gen AI SDK の統一インターフェースのみを使用します。旧来の `google-generativeai` パッケージや `vertexai.generative_models` 系の API は使いません。

## 前提条件

本ノートブックを実行する前に、ターミナルで以下を実行し、Application Default Credentials (ADC) を設定しておいてください。API キーは使用せず、認証は ADC に一本化します。

```bash
gcloud auth application-default login
```

また、リポジトリ直下の `.env` ファイルに `GOOGLE_CLOUD_PROJECT` (対象の Google Cloud プロジェクト ID) と `GOOGLE_CLOUD_LOCATION` (省略時は `global`) を設定しておいてください。

## トラブルシューティングのヒント

うまく動かない場合は、以下を順に確認してください。

- **認証エラーになる場合**: `gcloud auth application-default login` をやり直してください。ブラウザでのログインが完了しないまま放置されていることがあります。
- **`GOOGLE_CLOUD_PROJECT` が空、または誤ったプロジェクトになっていないか**: `.env` の内容と、`gcloud config get-value project` で確認できる現在の設定を見比べてください。
- **対象プロジェクトで Vertex AI API が有効化されているか**: Google Cloud コンソールで `aiplatform.googleapis.com` (Vertex AI API) が有効になっているか確認してください。
- **エラーの見分け方**: このノートブックのコードは `google.api_core.exceptions` 由来の例外を送出することがあります。`PermissionDenied` や `Unauthenticated` は認証・権限まわりの設定ミスであることが多く、`ResourceExhausted` (HTTP 429) はクォータ超過の可能性が高いです。まずは例外メッセージと HTTP ステータスコードを確認して、どちらの系統の問題かを切り分けましょう。


## 2. セットアップ

このセクションでは、`google-genai` パッケージのインストールから `.env` の読み込み、Vertex AI 経由で使用する `genai.Client` の初期化までを行います。以降のすべてのセルで使い回す `MODEL_ID` もここで定義します。

In [ ]:
import sys

# このリポジトリは uv でパッケージを管理しており、uv が作る .venv には
# (uv の設計上) pip 本体がインストールされていません。そのため %pip install や
# !pip install はどちらも "No module named pip" になり失敗します。
# uv 管理の venv では、pip 互換のインターフェースを持つ "uv pip install" を使い、
# --python でこのノートブックのカーネルが使っている Python を明示的に指定して
# インストールします(uv 自体がインストールされていることが前提です)。
!uv pip install --quiet --python {sys.executable} google-genai python-dotenv pillow

# 参考: uv を使わない一般的な Python 環境 (pip が最初から入っている venv など) では、
# 代わりに次の行のコメントを外して実行してください。
# %pip install --upgrade --quiet google-genai python-dotenv pillow

In [ ]:
import os
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types

# .env ファイルから GOOGLE_GENAI_USE_VERTEXAI / GOOGLE_CLOUD_PROJECT / GOOGLE_CLOUD_LOCATION
# などの環境変数を読み込みます。
load_dotenv()

print("google-genai の import が完了しました。")

In [ ]:
# vertexai=True を指定することで、Gemini Enterprise Agent Platform (旧 Vertex AI) 経由で
# モデルを呼び出します。認証情報は明示的に渡さず、ADC (gcloud auth application-default login
# で設定済みのもの) を利用します。
client = genai.Client(
    vertexai=True,
    project=os.environ["GOOGLE_CLOUD_PROJECT"],
    location=os.environ.get("GOOGLE_CLOUD_LOCATION", "global"),
)

# このノートブック内のテキスト系サンプルでは、以降すべてこの MODEL_ID を使い回します。
MODEL_ID = "gemini-3.7-flash"

print(f"クライアントを初期化しました。使用モデル: {MODEL_ID}")

## 3. 基本のテキスト生成

このセクションでは、`client.models.generate_content()` を使った最も基本的なテキスト生成の呼び出し方と、レスポンスオブジェクト (`GenerateContentResponse`) に含まれる情報の読み方を学びます。

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="生成AIの学習を始める初心者に向けて、励みになる一言をください。",
)

print(response.text)

# 参考: ここで google.api_core.exceptions 由来の例外が発生することがあります。
# - PermissionDenied / Unauthenticated -> ADC の認証設定が問題である可能性が高いです。
# - ResourceExhausted (HTTP 429)      -> クォータ超過の可能性が高いです。
# 例外メッセージと HTTP ステータスコードを見て、どちらの系統かを切り分けてください。

In [ ]:
# レスポンスオブジェクト (GenerateContentResponse) には、生成テキスト以外にも
# トークン使用量やモデルのバージョンなどの情報が含まれています。
usage = response.usage_metadata
print("=== usage_metadata ===")
print("prompt_token_count:", usage.prompt_token_count)
print("candidates_token_count:", usage.candidates_token_count)
print("total_token_count:", usage.total_token_count)

print()
print("=== レスポンスの基本情報 ===")
print("model_version:", response.model_version)
print("candidates の件数:", len(response.candidates))
print("1件目の finish_reason:", response.candidates[0].finish_reason)

## 4. マルチモーダル入力

このセクションでは、テキストに加えて画像を入力として渡し、画像の内容を踏まえた応答を得る方法を学びます。`contents` にはテキストと `types.Part` を混在させたリストを渡すことができます。

In [ ]:
# Google Cloud が公式サンプルとして公開している Cloud Storage 上の画像を URI で渡す例です。
# 補足: 当初 Wikimedia Commons などの一般公開 HTTPS URL で試すと、Vertex AI 側の URL
# 取得ポリシー (robots.txt の尊重やクロール制限) によって
# "Cannot fetch content from the provided URL" (400 INVALID_ARGUMENT) になることがあります。
# gs:// (Cloud Storage) の URI であればこの問題が起きず安定して動作するため、
# 本ノートブックでは gs:// URI を使う方法を基本形として採用しています。
image_uri = "gs://cloud-samples-data/generative-ai/image/scones.jpg"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        "この画像に写っている食べ物の様子を、日本語で2〜3文で説明してください。",
        types.Part.from_uri(file_uri=image_uri, mime_type="image/jpeg"),
    ],
)

print(response.text)

# 代替方法: ローカルにある画像ファイルを渡したい場合は、bytes を読み込んで
# types.Part.from_bytes(data=..., mime_type=...) を使います。
#
# with open("local_image.jpg", "rb") as f:
#     image_bytes = f.read()
#
# response = client.models.generate_content(
#     model=MODEL_ID,
#     contents=[
#         types.Part.from_text(text="この画像に写っているものを説明してください。"),
#         types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"),
#     ],
# )
# print(response.text)

## 5. マルチターン会話

このセクションでは、`client.chats.create()` で会話セッションを作成し、`chat.send_message()` を複数回呼び出すことで、モデルが会話の文脈を保持したまま応答することを確認します。

In [ ]:
chat = client.chats.create(model=MODEL_ID)

response1 = chat.send_message("こんにちは。生成AIの研修についてこれから質問します。")
print("--- 1ターン目 ---")
print(response1.text)

response2 = chat.send_message("今何について話すと予告しましたか？一言で教えてください。")
print("--- 2ターン目 ---")
print(response2.text)

response3 = chat.send_message("ありがとうございました。これで会話を終わります。")
print("--- 3ターン目 ---")
print(response3.text)

print()
print("=== 会話履歴 (get_history) ===")
for content in chat.get_history():
    print(f"[{content.role}] {content.parts[0].text}")

## 6. ストリーミング

このセクションでは、`client.models.generate_content_stream()` を使って、モデルの応答をチャンク単位で逐次受け取りながら表示する方法を学びます。チャンクによっては `text` が `None` になることがあるため、必ずガードして扱います。

In [ ]:
for chunk in client.models.generate_content_stream(
    model=MODEL_ID,
    contents="1から5までカウントアップしながら、それぞれの数字にまつわる豆知識を一言添えてください。",
):
    # chunk.text は None になることがあるため (candidates が空、
    # parts が function_call のみで構成されている場合など)、必ずガードします。
    if chunk.text is not None:
        print(chunk.text, end="", flush=True)

print()

## 7. GenerationConfig (生成パラメータの調整)

このセクションでは、`types.GenerateContentConfig` を使って `temperature` / `top_p` / `max_output_tokens` / `stop_sequences` といった生成パラメータを調整する方法を学びます。

> **重要**: Gemini 3 系のモデルでは、`temperature` / `top_p` / `top_k` を既定値から変更することは**非推奨**です。既定値のまま使うことが推奨されており、値を下げるとかえって同じ内容を繰り返す (ループする) などの異常動作の原因になり得ます。以下のコードは Gemini 2.5 系以前でよく使われていた調整テクニックを**参考として**紹介するものであり、Gemini 3 系のモデルに対して積極的に適用することは推奨しません。

In [ ]:
# 注意: temperature / top_p の変更は Gemini 2.5 系以前向けの旧テクニックです。
# Gemini 3 系 (本ノートブックの MODEL_ID) では既定値のままの利用が推奨されており、
# ここでは動作確認・参考のためにあえて変更しています。
config = types.GenerateContentConfig(
    temperature=0.2,
    top_p=0.95,
    max_output_tokens=256,
    stop_sequences=["###"],
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents="生成AIを使ったアプリ開発のアイデアを1つ、簡潔に提案してください。",
    config=config,
)

print(response.text)

## 8. 構造化出力 (Controlled Generation)

このセクションでは、pydantic の `BaseModel` でスキーマを定義し、`types.GenerateContentConfig(response_mime_type="application/json", response_schema=...)` を指定することで、モデルの出力を指定したスキーマに沿った JSON として取得する方法を学びます。パースされたオブジェクトは `response.parsed` から直接取得できます。

In [ ]:
import pydantic


class Recipe(pydantic.BaseModel):
    name: str
    ingredients: list[str]
    steps: list[str]


response = client.models.generate_content(
    model=MODEL_ID,
    contents="簡単な野菜炒めのレシピを教えてください。",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Recipe,
    ),
)

# response.parsed には、すでに Recipe インスタンスとしてパース済みのオブジェクトが入ります。
recipe = response.parsed
print("料理名:", recipe.name)
print("材料:", recipe.ingredients)
print("手順:", recipe.steps)

## 9. Function Calling

このセクションでは、Python 関数をそのまま `tools` に渡すだけでモデルが「関数を呼ぶべきか」を判断し、SDK が実行まで自動化してくれる**自動関数呼び出し (Automatic Function Calling, AFC)** を中心に学びます。あわせて、`types.FunctionDeclaration` を自分で定義して手動でハンドリングする方式も簡単に紹介します。

In [ ]:
def get_weather(city: str, unit: str = "celsius") -> str:
    """指定した都市の現在の天気を取得します。

    Args:
        city: 天気を調べたい都市名。
        unit: 温度の単位 (celsius または fahrenheit)。
    """
    # デモ用のダミーデータを返します。外部 API へのアクセスは行いません。
    return f"{city} は現在晴れ、気温は 25 {unit} です。"


# 素の Python 関数をそのまま tools に渡すことで、AFC (自動関数呼び出し) が有効になります。
# types.Tool でラップしてしまうと AFC 非対応と判定されるため、関数はラップせずに渡します。
chat = client.chats.create(
    model=MODEL_ID,
    config=types.GenerateContentConfig(tools=[get_weather]),
)

response = chat.send_message("東京の天気を教えてください。")
print(response.text)


def summarize_part(part):
    # 会話履歴の各 Part には text / function_call / function_response のいずれか
    # 1つだけが入っているため、どれが入っているかを見て要点だけを取り出します。
    if part.function_call is not None:
        return f"function_call: {part.function_call.name}({part.function_call.args})"
    if part.function_response is not None:
        return f"function_response: {part.function_response.name} -> {part.function_response.response}"
    return part.text


print()
print("=== 自動関数呼び出しを含む会話履歴 (chat.get_history()) ===")
# 補足: chat.send_message() 経由の AFC では、レスポンス側の
# automatic_function_calling_history は使えません (常に None になります)。
# これは client.models.generate_content() を直接呼んだ場合専用のフィールドで、
# Chat クラスは関数呼び出しのループを自前で実装しているためです。
# 関数呼び出し (function_call) とその結果 (function_response) のやり取りは、
# 代わりに chat.get_history() から確認できます。
for content in chat.get_history():
    print(f"[{content.role}]", summarize_part(content.parts[0]))

### 補足: 手動での Function Calling

`types.Tool(function_declarations=[...])` のように `types.Tool` で明示的にラップした場合は AFC の対象外となり、モデルからの `function_call` を自分で受け取って実行し、結果を送り返す必要があります。API 連携を細かく制御したい場合に使う方式です。


In [ ]:
get_weather_decl = types.FunctionDeclaration(
    name="get_weather",
    description="指定した都市の天気を取得します。",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={"city": types.Schema(type=types.Type.STRING, description="都市名")},
        required=["city"],
    ),
)
tool = types.Tool(function_declarations=[get_weather_decl])
config = types.GenerateContentConfig(tools=[tool])

response = client.models.generate_content(
    model=MODEL_ID,
    contents="大阪の天気を教えてください。",
    config=config,
)

# types.Tool でラップした場合は自動実行されないため、function_calls から
# モデルが呼びたい関数名と引数を自分で取り出します。
call = response.function_calls[0]
print("呼び出したい関数:", call.name)
print("引数:", call.args)

# 実際の関数呼び出し (ここではダミーの文字列を組み立てるだけ)。
result_text = f"{call.args['city']} は現在晴れ、気温は 25 celsius です。"
function_response_part = types.Part.from_function_response(
    name=call.name,
    response={"result": result_text},
)

# function_call を含むモデルの応答と、function_response を会話履歴に積んで再度呼び出します。
contents = [
    types.Content(role="user", parts=[types.Part.from_text(text="大阪の天気を教えてください。")]),
    response.candidates[0].content,
    types.Content(role="user", parts=[function_response_part]),
]
final_response = client.models.generate_content(
    model=MODEL_ID,
    contents=contents,
    config=config,
)
print()
print("最終応答:", final_response.text)

## 10. Grounding (Google 検索によるグラウンディング)

このセクションでは、`types.Tool(google_search=types.GoogleSearch())` を `tools` に渡すことで、モデルが必要に応じて Google 検索を行い、最新の事実に基づいた回答を生成する Grounding 機能を学びます。

> **重要**: Google 検索でグラウンディングされた応答を画面に表示する場合、Google の利用ポリシーにより **Search Suggestion (検索候補) の表示が義務付けられています**。`grounding_metadata.search_entry_point.rendered_content` に表示用の HTML が含まれるため、実際にエンドユーザー向けの UI で表示する際は、必ずこの Search Suggestion もあわせて表示してください。本ノートブックでは学習目的のため表示ロジックは簡略化していますが、実運用ではこの点を省略しないよう注意してください。

In [ ]:
config = types.GenerateContentConfig(
    tools=[types.Tool(google_search=types.GoogleSearch())],
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents="直近の生成AI関連のニュースを1つ教えてください。",
    config=config,
)

print(response.text)

print()
print("=== グラウンディング情報 ===")
grounding_metadata = response.candidates[0].grounding_metadata
if grounding_metadata is not None:
    print("検索クエリ:", grounding_metadata.web_search_queries)
    for chunk in grounding_metadata.grounding_chunks or []:
        if chunk.web is not None:
            print("-", chunk.web.title, chunk.web.uri)
    # 上記のとおり、実際に画面へ表示する際は search_entry_point.rendered_content
    # (Search Suggestion) の表示も忘れないようにしてください。

## 11. 画像生成・編集 (Nano Banana)

このセクションでは、画像出力に対応したモデル (通称 "Nano Banana") を使って、テキストから画像を生成する方法と、生成した画像に対して追加の指示を与えて対話的に編集する方法を学びます。

In [ ]:
import io

from PIL import Image

response = client.models.generate_content(
    model="gemini-3.1-flash-image",
    contents="白猫がひなたぼっこをしている、あたたかい雰囲気のイラストを生成してください。",
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
    ),
)

first_image_bytes = None
first_image_mime_type = None
for part in response.candidates[0].content.parts:
    if part.inline_data is not None:
        first_image_bytes = part.inline_data.data
        first_image_mime_type = part.inline_data.mime_type
        image = Image.open(io.BytesIO(first_image_bytes))
        display(image)
    elif part.text is not None:
        print(part.text)

In [ ]:
# 前のセルで生成した画像 (inline_data) を、そのまま次のリクエストの入力として渡すことで、
# 対話的に画像を編集できます。
edited_response = client.models.generate_content(
    model="gemini-3.1-flash-image",
    contents=[
        types.Part.from_bytes(data=first_image_bytes, mime_type=first_image_mime_type),
        "背景を夜景に変更してください。",
    ],
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
    ),
)

for part in edited_response.candidates[0].content.parts:
    if part.inline_data is not None:
        edited_image = Image.open(io.BytesIO(part.inline_data.data))
        display(edited_image)
    elif part.text is not None:
        print(part.text)

# 使い分けの目安:
# - "gemini-3.1-flash-image" (Nano Banana 2): 低レイテンシ・低コストで、素早い試行錯誤や
#   プロトタイピングに向いています。
# - "gemini-3-pro-image" (Nano Banana Pro): より高品質な画像が必要な本番用途向けのモデルです。
# Vertex AI 経由では、モデル ID の末尾に "-preview" サフィックスが付くバージョンが
# 提供される場合があります。実際に使用する際は、利用可能なモデル ID を Google Cloud
# コンソールやドキュメントで確認してください。

## 12. 動画生成 (Veo 3.1)

このセクションでは、Veo 3.1 モデルによる動画生成を学びます。動画生成は非同期のオペレーションとして実行されるため、`operation.done` が `True` になるまでポーリングして完了を待つ処理が必要です。

> **重要**: 動画生成には数分程度かかる場合があり、実行すると課金が発生します。実際に実行する際は、時間と費用を考慮したうえで行ってください。

In [ ]:
operation = client.models.generate_videos(
    model="veo-3.1-generate-001",
    # 新しい google-genai SDK では、generate_videos に prompt/image/video を直接
    # 渡す呼び出し方は非推奨 (Deprecated) になっています。types.GenerateVideosSource
    # でラップして source 引数に渡す形が現在の正しい書き方です。
    source=types.GenerateVideosSource(
        prompt="黒猫が高速で走り抜けるネオンホログラム風の映像。",
    ),
    config=types.GenerateVideosConfig(
        aspect_ratio="9:16",
        duration_seconds=8,
        number_of_videos=1,
        generate_audio=True,
    ),
)

print("動画生成オペレーションを開始しました。完了までポーリングします...")

# 動画生成は非同期オペレーションのため、done になるまでポーリングして待ちます。
while not operation.done:
    time.sleep(10)
    operation = client.operations.get(operation)

print("動画生成が完了しました。")

generated_videos = operation.result.generated_videos
video = generated_videos[0].video

# GenerateVideosConfig で output_gcs_uri を指定しなかった場合、動画データは
# base64 で埋め込まれて返り、video.video_bytes にデコード済みのバイト列が入ります。
video.save("m1_generated_video.mp4")
print("動画をローカルに保存しました: m1_generated_video.mp4")

from IPython.display import Video

Video("m1_generated_video.mp4")

# Vertex AI 経由では、モデル ID の末尾に "-preview" サフィックスが付くバージョンが
# 提供される場合があります。実際に使用する際は、利用可能なモデル ID をご確認ください。

## 13. まとめ

このノートブックで扱った、Google Gen AI SDK の主な API 呼び出しパターンを一覧にまとめます。

| セクション | 目的 | 主な API |
| --- | --- | --- |
| 基本のテキスト生成 | 単発のテキスト生成 | `client.models.generate_content()` |
| マルチモーダル入力 | 画像+テキストの入力 | `types.Part.from_uri()` / `types.Part.from_bytes()` |
| マルチターン会話 | 文脈を保持した対話 | `client.chats.create()` / `chat.send_message()` |
| ストリーミング | 応答の逐次表示 | `client.models.generate_content_stream()` |
| GenerationConfig | 生成パラメータの調整 | `types.GenerateContentConfig(temperature=..., top_p=...)` |
| 構造化出力 | JSON スキーマに沿った出力 | `response_mime_type` / `response_schema` / `response.parsed` |
| Function Calling | 外部関数の自動/手動呼び出し | `tools=[関数]` (AFC) / `types.FunctionDeclaration` (手動) |
| Grounding | Google 検索に基づく回答 | `types.Tool(google_search=types.GoogleSearch())` |
| 画像生成・編集 | Nano Banana によるテキスト→画像・画像編集 | `generate_content(model="gemini-3.1-flash-image", ...)` |
| 動画生成 | Veo 3.1 による動画生成 | `client.models.generate_videos()` / `client.operations.get()` |

Gemini Enterprise Agent Platform 上では、これらすべてを共通の `genai.Client` / `google.genai.types` インターフェースで扱えることが、Google Gen AI SDK の大きな特徴です。モデルや機能ごとに異なる SDK を使い分ける必要がなく、`config` に渡すオブジェクトを差し替えるだけで、テキスト・マルチモーダル・構造化出力・Function Calling・Grounding・画像/動画生成まで一貫した書き方で実装できます。